In [1]:
# Import necessary libraries
import os
import glob
import json

import numpy as np
from IPython.display import display, Image
import ipywidgets as widgets
from ipywidgets import HBox, VBox

from PIL import Image, ImageDraw


In [11]:
BASE_FOLDER_PTH = "../../../../data/people_detection"
# DATASET_NAME = "filtered_v1_10_sec"
DATASET_NAME = "baseline_10sec"
BASE_PTH = os.path.join(BASE_FOLDER_PTH, DATASET_NAME)

SUBSET = "holdout"
IMGS_PTH = os.path.join(BASE_PTH, "images", SUBSET)
LABELS_PTH = os.path.join(BASE_PTH, "labels", SUBSET)

img_paths = np.array(sorted(glob.glob(os.path.join(IMGS_PTH, "*"))))
label_paths = np.array(sorted(glob.glob(os.path.join(LABELS_PTH, "*"))))

log_pth = f"log_{DATASET_NAME}_{SUBSET}.json"

video_names = [os.path.basename(i)[:-10] for i in img_paths]
uvideo_names, inverse_mapping = np.unique(video_names, return_inverse=True)

video_frames = []
np.random.seed(42)
for i in range(len(uvideo_names)):
    idxs = np.where(inverse_mapping == i)[0]
    np.random.shuffle(idxs)
    
    video_frames.append((
        uvideo_names[i],
        img_paths[idxs[:2]].tolist(),
        label_paths[idxs[:2]].tolist(),
    ))

is_filtered = [[False, False, False] for _ in range(len(video_frames))]
is_viewed = [False] * len(video_frames)
# Initialize the index for the current image
current_index = 0

In [12]:
img_paths

array(['../../../../data/people_detection/baseline_10sec/images/holdout/2.12.21_H265_hall_3_N00003_00100.jpg',
       '../../../../data/people_detection/baseline_10sec/images/holdout/2.12.21_H265_hall_3_N00003_00500.jpg',
       '../../../../data/people_detection/baseline_10sec/images/holdout/2.12.21_H265_hall_3_N00003_00800.jpg',
       ...,
       '../../../../data/people_detection/baseline_10sec/images/holdout/pg1_Dev_H220_MR_NY_1509275838_01700.jpg',
       '../../../../data/people_detection/baseline_10sec/images/holdout/pg1_Dev_H220_MR_NY_1509275838_01800.jpg',
       '../../../../data/people_detection/baseline_10sec/images/holdout/pg1_Dev_H220_MR_NY_1509275838_01900.jpg'],
      dtype='<U148')

In [13]:
if os.path.exists(log_pth):
    with open (log_pth, "rt") as f:
        log_data = json.load(f)

        is_viewed = log_data["is_viewed"]
        is_filtered = log_data["is_filtered"]
        current_index = log_data["current_index"]
        
else:

    log_data = {
        "current_index": current_index,
        "video_frames": video_frames,
        "is_viewed": is_viewed, 
        "is_filtered": is_filtered,
    }
    

def log_state():
    log_data = {
        "current_index": current_index,
        "is_viewed": is_viewed, 
        "is_filtered": is_filtered,
    }
    with open(log_pth, "wt") as f:
        json.dump(log_data, f, indent=4)

In [14]:
def get_concat_h(im1, im2):
    dst = Image.new('RGB', (im1.width + im2.width, im1.height))
    dst.paste(im1, (0, 0))
    dst.paste(im2, (im1.width, 0))
    return dst

In [15]:
def draw_gt(im, label_pth):
    img_w, img_h = im.size
    draw = ImageDraw.Draw(im)
    
    with open(label_pth, "rt") as f:
        for line in f.readlines():
            _, x_c, y_c, w, h, x, y = [float(i) for i in line.split()]
            x_c *= img_w
            y_c *= img_h
            w *= img_w
            h *= img_h
            box = [int(x_c - w / 2), int(y_c - h / 2), int(x_c + w / 2), int(y_c + h / 2)]

            draw.rectangle(box, None, (0, 255, 0), 2)
            draw.circle([int(x * img_w), int(y * img_h)], 3, (0, 255, 0))


def resize_im(im, tagget_w):
    w, h = im.size
    scale = tagget_w / w
    w = int(scale * w)
    h = int(scale * h)
    im = im.resize((w, h))
    
    return im

In [16]:
TARGET_W = 700

# Create a Label widget to display the image name
video_name_label = widgets.Label()



# Function to display the current image
def show_image(index):
    
    # img = cv2.imread(image_paths[index])
    # plt.imshow(img)
    im0 = Image.open(video_frames[index][1][0])
    im0 = resize_im(im0, TARGET_W)
    draw_gt(im0, video_frames[index][2][0])

    if len(video_frames[index][1]) > 1:
        im1 = Image.open(video_frames[index][1][1])
        im1 = resize_im(im1, TARGET_W)
        draw_gt(im1, video_frames[index][2][1])
        im0 = get_concat_h(im0, im1)
    
    display(im0)

# Function to handle button clicks
def on_button_click(b):
    global current_index

    if current_index == len(video_frames) - 1:
        is_viewed[current_index] = True
        return

    if b.description == 'Next':
        is_viewed[current_index] = True
        # log_state()
        current_index = current_index + 1
    output.clear_output()
    update_progress_bar()
    # Update the image name label
    video_name_label.value = video_frames[current_index][0]

    with output:
        show_image(current_index)


def on_filter_click(b):
    global current_index
    if b.description == "Filter size":
        is_filtered[current_index][1] = True
    
    if b.description == "Filter box":
        is_filtered[current_index][0] = True
        
    if b.description == "Filter pos":
        is_filtered[current_index][2] = True

# Function to update the progress bar
def update_progress_bar():
    progress.value = current_index + 1
    progress.description = f'{current_index + 1} / {len(video_frames)}'

# Create Previous and Next buttons
next_button = widgets.Button(description='Next')
filter_button0 = widgets.Button(description='Filter box')
filter_button1 = widgets.Button(description='Filter size')
filter_button2 = widgets.Button(description='Filter pos')

# Attach the button click event handlers
next_button.on_click(on_button_click)
filter_button0.on_click(on_filter_click)
filter_button1.on_click(on_filter_click)
filter_button2.on_click(on_filter_click)

# Create an output widget to display the image
output = widgets.Output()

# Create a progress bar
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(video_frames),
    description='0 / 0',
    bar_style='info',
    style={'description_width': 'initial'}
)

# Display the initial image and update the progress bar
update_progress_bar()
video_name_label.value = video_frames[current_index][0]
with output:
    show_image(current_index)


# Arrange the buttons, progress bar, and output in a vertical box
ui = VBox([HBox([next_button, filter_button0, filter_button1, filter_button2]), progress, video_name_label, output])

# Display the UI
display(ui)